In [143]:
import fastf1
import pandas as pd
import numpy as np
import os

cache_dir = "cache/cache_data"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

fastf1.Cache.enable_cache(cache_dir)

session = fastf1.get_session(2023, "Monaco", "R") # Race
session.load()

laps = session.laps # DataFrame with cols
weather = session.weather_data

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '14', '31', '44', '63', '16', '10', '55', '4', '81', '77', '21', '24', '23', '22', '11', '27', '2', '20', '18']


In [144]:
laps.columns.tolist()

['Time',
 'Driver',
 'DriverNumber',
 'LapTime',
 'LapNumber',
 'Stint',
 'PitOutTime',
 'PitInTime',
 'Sector1Time',
 'Sector2Time',
 'Sector3Time',
 'Sector1SessionTime',
 'Sector2SessionTime',
 'Sector3SessionTime',
 'SpeedI1',
 'SpeedI2',
 'SpeedFL',
 'SpeedST',
 'IsPersonalBest',
 'Compound',
 'TyreLife',
 'FreshTyre',
 'Team',
 'LapStartTime',
 'LapStartDate',
 'TrackStatus',
 'Position',
 'Deleted',
 'DeletedReason',
 'FastF1Generated',
 'IsAccurate']

### **DataFrame Cleaning**

In [145]:
laps['TyreLife'].head()
df = laps.copy()

df = df.dropna(subset=['LapTime'])

df['LapTime'] = df['LapTime'].dt.total_seconds()

### **Filter out non-representative laps**
- make sure to explore the race_control_message for better model fit

In [146]:
# remove lap 1 of each race
df = df[df['LapNumber'] != 1]

# remove in-laps & out laps (lap where driver pits and after pitting)
df = df[df['PitInTime'].isna()] # remove in-laps
df = df[df['PitOutTime'].isna()] # remove out-laps

# remove laps under safety car, yellow, red, flag
df = df[df['TrackStatus'] == '1'] # keeping only clean laps

# flag laps where gap ahead < 1.5s (unsure how to do this)

### **Fuel Correction per lap**


In [147]:
df['LapNumber'].max()

np.float64(78.0)

### **Fuel Compenstation**
- we want to compensate the fuel loss throughout the race as a normalization technique. 
- minimize the influence of lap times on tyre degredation

In [148]:
starting_fuel = 110 # est in kg
total_race_laps = session.total_laps
fuel_per_lap = starting_fuel / total_race_laps
fuel_time_per_kg = 0.035 # s/kg


# laps_remaining = total_race_laps - df['LapNumber'] 
fuel_correction = df['LapNumber'] * fuel_per_lap * fuel_time_per_kg # how much fuel has been burned, the advantage to compensate for

df['FuelCorrectedLapTime'] = df['LapTime'] + fuel_correction

/var/folders/mp/5j6y5_ns2cx879p6y44k_6000000gn/T/ipykernel_94466/3531198020.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['FuelCorrectedLapTime'] = df['LapTime'] + fuel_correction


### **Dirty air Flagging**
Concept:
- When an F1 car is within roughly 1.5 seconds of the car ahead, it loses aerodynamic downforce from the turbulent air (dirty air or wake). This makes the car slower, particularly in speed corners. The lap time is artifically inflated, not because of tires, but because of aerodynamics. We want to flag these laps so we can either exclude them from the degradation model or treat them separately.

Pseudocode:
```
    For each lap_number in the race:
        1. Get all drivers' data for that lap
        2. Sort by Position (track position, P1 first)
        3. For each driver (except P1):
            - Find the driver one position ahead
            - gap = this_driver.LapStartDate - ahead_driver.LapStartDate
            - Convert gap to seconds
            - If gap < 1.5s → flag as dirty air
        4. P1 is never in dirty air (no one ahead)
```


In [150]:
df.sort_values(['LapNumber', 'Position']).groupby('LapNumber')['LapStartDate'].shift(1)

1                          NaT
233    2023-05-28 13:04:28.412
973    2023-05-28 13:04:29.701
1206   2023-05-28 13:04:30.813
1128   2023-05-28 13:04:31.564
                 ...          
1204   2023-05-28 14:51:08.913
1360   2023-05-28 14:51:10.111
387    2023-05-28 14:51:20.229
155    2023-05-28 14:51:31.105
1282   2023-05-28 14:51:31.938
Name: LapStartDate, Length: 1259, dtype: datetime64[ns]

In [ ]:
df['FuelCorrectedLapTime']

1       79.465718
2       79.222077
3       78.326436
4       78.265795
5       77.936154
          ...    
1510    91.046205
1511    89.837564
1512    89.844923
1513    89.366282
1514    88.585641
Name: FuelCorrectedLapTime, Length: 1259, dtype: float64

In [ ]:
df['LapTime']

1       79.367
2       79.074
3       78.129
4       78.019
5       77.640
         ...  
1510    87.443
1511    86.185
1512    86.143
1513    85.615
1514    84.785
Name: LapTime, Length: 1259, dtype: float64

In [ ]:
weather

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:42.422000,25.8,39.0,1013.7,False,47.4,149,1.0
1,0 days 00:01:42.421000,25.8,40.0,1013.7,False,48.2,158,1.0
2,0 days 00:02:42.419000,25.8,41.0,1013.7,False,48.2,137,1.0
3,0 days 00:03:42.418000,25.8,41.0,1013.7,False,48.3,0,0.8
4,0 days 00:04:42.433000,25.8,41.0,1013.7,False,48.3,0,0.5
...,...,...,...,...,...,...,...,...
171,0 days 02:51:43.083000,23.9,56.0,1012.9,False,30.7,134,1.3
172,0 days 02:52:43.097000,23.9,54.0,1013.0,False,30.7,131,1.1
173,0 days 02:53:43.096000,23.9,55.0,1012.7,False,30.5,115,0.6
174,0 days 02:54:43.095000,23.9,56.0,1012.7,False,30.6,135,0.5
